# Tema 5 — Listado II — Ejercicio 2
# Inferencia con BERT sobre textos nuevos

En este notebook se resuelve el **Ejercicio 2 del Listado II del Tema 5**.

El ejercicio pide realizar lo mismo que en el Ejercicio 1, pero ahora haciendo inferencia sobre frases nuevas:

```text
'i hate you too much'
'I did not like the film at all'
'I loved the movie'
```

La idea es:

1. Cargar el dataset `rotten_tomatoes`.
2. Tokenizarlo con el tokenizer de `bert-base-uncased`.
3. Hacer fine-tuning de BERT para clasificación binaria.
4. Crear una función de inferencia para textos nuevos.
5. Tokenizar las frases nuevas con el mismo tokenizer.
6. Usar `truncation=True`.
7. Obtener la clase predicha y sus probabilidades.

Las clases son:

- `neg`: comentario negativo.
- `pos`: comentario positivo.


## 0. Instalación de librerías

En Google Colab puedes ejecutar esta celda si no tienes instaladas las librerías.

Si trabajas en local con `uv`, puedes usar:

```bash
uv add transformers datasets evaluate accelerate scikit-learn torch pandas
```


In [1]:
# En Colab, descomenta esta línea si hace falta:
# !pip install -q transformers[torch] datasets evaluate accelerate scikit-learn pandas

## 1. Importación de librerías

Usaremos:

- `datasets` para cargar `rotten_tomatoes`.
- `transformers` para BERT, tokenizer, `Trainer` y `TrainingArguments`.
- `sklearn` para calcular métricas.
- `torch` para aplicar `softmax`.
- `pandas` para mostrar los resultados en tabla.


In [2]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Semillas para reproducibilidad aproximada
np.random.seed(42)
torch.manual_seed(42)

print("Versión de torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

Versión de torch: 2.12.0+cpu
CUDA disponible: False


## 2. Cargar el dataset `rotten_tomatoes`

El dataset ya viene dividido en:

- `train`
- `validation`
- `test`

Las labels son:

- `0`: `neg`
- `1`: `pos`


In [3]:
dataset = load_dataset("rotten_tomatoes")

print(dataset)

labels = dataset["train"].features["label"].names
NUM_LABELS = len(labels)

print("\nLabels:", labels)
print("Número de labels:", NUM_LABELS)

print("\nEjemplo:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Labels: ['neg', 'pos']
Número de labels: 2

Ejemplo:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


## 3. Cargar el tokenizer de BERT

Usamos el mismo modelo del Ejercicio 1:

```python
bert-base-uncased
```

Este tokenizer convierte los textos en:

- `input_ids`
- `attention_mask`

Además, para las frases nuevas también deberemos usar este mismo tokenizer con `truncation=True`.


In [4]:
model_id = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Tokenizer cargado:", model_id)
print("Longitud máxima admitida por BERT:", tokenizer.model_max_length)

Tokenizer cargado: bert-base-uncased
Longitud máxima admitida por BERT: 512


## 4. Elegir longitud máxima

BERT admite hasta 512 tokens, pero el dataset `rotten_tomatoes` tiene frases cortas.

Calculamos la longitud máxima en el conjunto de entrenamiento para decidir el valor de `MAX_LENGTH`.

También podríamos fijar `MAX_LENGTH = 128`, que suele ser una opción cómoda.


In [5]:
token_lengths = [
    len(tokenizer(text).input_ids)
    for text in dataset["train"]["text"]
]

print("Longitud mínima:", np.min(token_lengths))
print("Longitud media:", np.mean(token_lengths))
print("Longitud máxima:", np.max(token_lengths))
print("Percentil 95:", np.percentile(token_lengths, 95))

MAX_LENGTH = max(token_lengths)

print("\nMAX_LENGTH elegido:", MAX_LENGTH)

Longitud mínima: 3
Longitud media: 27.368347010550995
Longitud máxima: 78
Percentil 95: 47.0

MAX_LENGTH elegido: 78


## 5. Tokenizar el dataset

Aplicamos padding y truncamiento:

```python
padding="max_length"
truncation=True
max_length=MAX_LENGTH
```

Esto es importante porque todas las entradas deben tener la misma longitud.


In [6]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

encoded_data = dataset.map(tokenize, batched=True)

encoded_data

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})

## 6. Crear subconjuntos pequeños

Para que el notebook sea ejecutable en poco tiempo, entrenamos con subconjuntos pequeños.

Puedes aumentar estos valores si usas GPU o quieres mejores resultados.


In [7]:
small_train_dataset = encoded_data["train"].shuffle(seed=42).select(range(1000))
small_validation_dataset = encoded_data["validation"].shuffle(seed=42).select(range(500))
small_test_dataset = encoded_data["test"].shuffle(seed=42).select(range(500))

columns_to_keep = ["input_ids", "attention_mask", "label"]

small_train_dataset.set_format(type="torch", columns=columns_to_keep)
small_validation_dataset.set_format(type="torch", columns=columns_to_keep)
small_test_dataset.set_format(type="torch", columns=columns_to_keep)

print("Small train:", len(small_train_dataset))
print("Small validation:", len(small_validation_dataset))
print("Small test:", len(small_test_dataset))

Small train: 1000
Small validation: 500
Small test: 500


## 7. Cargar BERT para clasificación

Usamos:

```python
AutoModelForSequenceClassification
```

Esta clase carga BERT y añade una cabeza de clasificación final.

Como tenemos dos clases, indicamos:

```python
num_labels=2
```


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=NUM_LABELS,
    id2label={0: "neg", 1: "pos"},
    label2id={"neg": 0, "pos": 1}
)

print("Modelo cargado correctamente.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado correctamente.


## 8. Métricas

Definimos las métricas que se usarán durante evaluación:

- Accuracy.
- Precision macro.
- Recall macro.
- F1 macro.


In [9]:
def compute_metrics(pred):
    y_true = pred.label_ids
    y_pred = pred.predictions.argmax(-1)

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## 9. Hiperparámetros

Configuramos el entrenamiento con `TrainingArguments`.

Usamos una configuración ligera para que el notebook pueda ejecutarse en Colab o en local.


In [10]:
try:
    args = TrainingArguments(
        output_dir="./outputs_ejercicio2",
        report_to="none",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=25
    )
except TypeError:
    args = TrainingArguments(
        output_dir="./outputs_ejercicio2",
        report_to="none",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=25
    )

args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval

## 10. Crear el `Trainer`

El `Trainer` recibe:

- Modelo.
- Hiperparámetros.
- Dataset de entrenamiento.
- Dataset de validación.
- Función de métricas.


In [11]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train_dataset,
    eval_dataset=small_validation_dataset,
    compute_metrics=compute_metrics
)

print("Trainer creado correctamente.")

Trainer creado correctamente.


## 11. Entrenar el modelo

Realizamos el fine-tuning.

Si tarda demasiado, puedes reducir:

```python
num_train_epochs=1
```

o usar menos ejemplos en `small_train_dataset`.


In [12]:
trainer.train()

C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.650970,0.503397,0.788000,0.787128,0.802480,0.793474
2,0.462078,0.409367,0.850000,0.849951,0.850506,0.851366


C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=64, training_loss=0.516283705830574, metrics={'train_runtime': 376.1356, 'train_samples_per_second': 5.317, 'train_steps_per_second': 0.17, 'total_flos': 80166649680000.0, 'train_loss': 0.516283705830574, 'epoch': 2.0})

## 12. Evaluación rápida sobre test

Antes de pasar a las frases nuevas, evaluamos el modelo sobre un subconjunto de test.

Esto nos da una idea de si el fine-tuning ha funcionado.


In [13]:
test_predictions = trainer.predict(small_test_dataset)

logits = test_predictions.predictions
y_true = test_predictions.label_ids

probabilities = torch.nn.functional.softmax(
    torch.tensor(logits),
    dim=-1
).numpy()

y_pred = np.argmax(probabilities, axis=-1)

print("Classification report sobre test:")
print(classification_report(
    y_true,
    y_pred,
    target_names=labels,
    zero_division=0
))

print("Matriz de confusión:")
print(confusion_matrix(y_true, y_pred))

C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Classification report sobre test:
              precision    recall  f1-score   support

         neg       0.85      0.79      0.82       263
         pos       0.78      0.84      0.81       237

    accuracy                           0.81       500
   macro avg       0.81      0.82      0.81       500
weighted avg       0.82      0.81      0.81       500

Matriz de confusión:
[[208  55]
 [ 38 199]]


# 13. Ejercicio 2 — Inferencia sobre textos nuevos

El enunciado pide probar las siguientes frases:

```text
'i hate you too much'
'I did not like the film at all'
'I loved the movie'
```

Muy importante: estas frases deben tokenizarse igual que el corpus de entrenamiento.

Además, el enunciado indica que hay que usar:

```python
truncation=True
```


In [14]:
new_texts = [
    "i hate you too much",
    "I did not like the film at all",
    "I loved the movie"
]

new_texts

['i hate you too much', 'I did not like the film at all', 'I loved the movie']

## 14. Función de inferencia para textos nuevos

La función realiza estos pasos:

1. Tokeniza el texto con el mismo tokenizer de BERT.
2. Usa `padding="max_length"`.
3. Usa `truncation=True`.
4. Pasa el texto por el modelo.
5. Aplica `softmax` para obtener probabilidades.
6. Aplica `argmax` para obtener la clase predicha.


In [15]:
def predict_sentiment(text):
    # Tokenización igual que en entrenamiento
    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Enviar tensores al mismo dispositivo que el modelo
    device = model.device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = labels[pred_id]

    return {
        "text": text,
        "pred_id": pred_id,
        "pred_label": pred_label,
        "prob_neg": float(probs[0]),
        "prob_pos": float(probs[1])
    }

## 15. Aplicar inferencia a las frases del ejercicio

Ahora aplicamos la función anterior a las tres frases nuevas.

Mostramos:

- Texto original.
- Clase predicha.
- Probabilidad de negativo.
- Probabilidad de positivo.


In [16]:
results = [predict_sentiment(text) for text in new_texts]

results_df = pd.DataFrame(results)

results_df

,text,pred_id,pred_label,prob_neg,prob_pos
0,i hate you too much,1,pos,0.488791,0.511209
1,I did not like the film at all,0,neg,0.697922,0.302078
2,I loved the movie,1,pos,0.305755,0.694245


## 16. Mostrar resultados en formato legible

Interpretamos las predicciones.

En principio, lo esperable sería:

| Frase | Resultado esperado |
|---|---|
| `i hate you too much` | Negativo |
| `I did not like the film at all` | Negativo |
| `I loved the movie` | Positivo |

Sin embargo, el primer texto no es exactamente una reseña de película, por lo que el modelo podría comportarse peor en esa frase.


In [17]:
for result in results:
    print("Texto:", result["text"])
    print("Predicción:", result["pred_label"])
    print(f"Probabilidad neg: {result['prob_neg']:.4f}")
    print(f"Probabilidad pos: {result['prob_pos']:.4f}")
    print("-" * 80)

Texto: i hate you too much
Predicción: pos
Probabilidad neg: 0.4888
Probabilidad pos: 0.5112
--------------------------------------------------------------------------------
Texto: I did not like the film at all
Predicción: neg
Probabilidad neg: 0.6979
Probabilidad pos: 0.3021
--------------------------------------------------------------------------------
Texto: I loved the movie
Predicción: pos
Probabilidad neg: 0.3058
Probabilidad pos: 0.6942
--------------------------------------------------------------------------------


## 17. Comprobación manual de acierto

Creamos una interpretación esperada para comparar con la salida del modelo.

Esto no es una evaluación formal, porque solo hay tres frases, pero sirve para responder a la pregunta:

> ¿Está acertando en los resultados?


In [18]:
expected_labels = ["neg", "neg", "pos"]

comparison_df = results_df.copy()
comparison_df["expected_label"] = expected_labels
comparison_df["correct"] = comparison_df["pred_label"] == comparison_df["expected_label"]

comparison_df

,text,pred_id,pred_label,prob_neg,prob_pos,expected_label,correct
0,i hate you too much,1,pos,0.488791,0.511209,neg,False
1,I did not like the film at all,0,neg,0.697922,0.302078,neg,True
2,I loved the movie,1,pos,0.305755,0.694245,pos,True


## 18. Conclusión automática

Generamos una pequeña conclusión según los aciertos obtenidos.


In [19]:
num_correct = comparison_df["correct"].sum()
total = len(comparison_df)

print(f"Aciertos: {num_correct}/{total}")

if num_correct == total:
    print("El modelo acierta las tres frases nuevas.")
elif num_correct == 0:
    print("El modelo no acierta ninguna de las frases nuevas.")
else:
    print("El modelo acierta algunas frases, pero no todas.")

print("\nComentario:")
print(
    "Las frases negativas deberían clasificarse como 'neg' y la frase 'I loved the movie' "
    "debería clasificarse como 'pos'. Si el modelo falla, puede deberse a que se ha entrenado "
    "con un subconjunto pequeño, pocas épocas o a que alguna frase no se parece exactamente "
    "a una reseña de película del dataset."
)

Aciertos: 2/3
El modelo acierta algunas frases, pero no todas.

Comentario:
Las frases negativas deberían clasificarse como 'neg' y la frase 'I loved the movie' debería clasificarse como 'pos'. Si el modelo falla, puede deberse a que se ha entrenado con un subconjunto pequeño, pocas épocas o a que alguna frase no se parece exactamente a una reseña de película del dataset.


## 19. Conclusión final

En este ejercicio se ha reutilizado el flujo de fine-tuning de BERT del Ejercicio 1 para hacer inferencia sobre textos nuevos.

La clave es que los textos nuevos deben pasar por el mismo proceso de tokenización que el dataset original:

```python
padding="max_length"
truncation=True
max_length=MAX_LENGTH
```

Después, el modelo devuelve logits, que se transforman en probabilidades mediante `softmax`. Finalmente, se aplica `argmax` para seleccionar la clase con mayor probabilidad.

Este procedimiento permite usar un modelo BERT ajustado para clasificar nuevas frases no vistas durante el entrenamiento.
